In [1]:
import pandas as pd 
import numpy as np 

In [3]:
df = pd.read_csv('tempDataset/final_dataset.csv')


In [4]:
import re
import json

def extract_json(text):
    # remove code fences if they exist
    text = re.sub(r"^```[a-zA-Z]*\n?", "", text.strip())  # remove leading ```json or ```
    text = re.sub(r"```$", "", text.strip())              # remove trailing ```
    
    # extract first JSON object if multiple
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group(0)
    return text


In [61]:
for i in range(3):
    resume = sample_df.iloc[i,0]
    job    = sample_df.iloc[i,1]

    prompt_content = f"""
    You are an expert resume analyst. Your task is to analyze a candidate's resume and a job description to determine their fit. You MUST ONLY provide a JSON object in your response. Do not include any other text or conversational phrases.

    [JOB DESCRIPTION]
    {job}

    [CANDIDATE RESUME]
    {resume}

    Analyze the candidate's skills and experience against the job description.
    Provide a final match score on a scale from 1 to 100, where 1 is no match and 100 is a perfect match.
    Then, provide a brief, professional summary explaining the score, highlighting the key strengths and weaknesses of the candidate relative to the job.

    Format your output strictly as a JSON object with the following keys:
    {{
      "match_score": int,
      "summary": "string"
    }}
    """

    response = ollama.chat(
        model='gemma3',
        messages=[{'role': 'user', 'content': prompt_content}]
    )

    try:
        raw_content = response['message']['content']
        json_str = extract_json(raw_content)   # 🧹 clean response
        ats_score_data = json.loads(json_str)
    
        match_score = ats_score_data.get('match_score', 'N/A')
        summary     = ats_score_data.get('summary', 'No summary provided.')
    
        atscoretemp.append(match_score)
        summarytemp.append(summary)
    
        print(i, 'done')
    
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON response at index {i}: {e}")
        print("Raw response:", response['message']['content'])


0 done
1 done
2 done


In [53]:
prev  = 0;
for i in range(100, 1600 , 100):
    new_df = df.loc[prev:i, : ]
    new_df.to_csv(f"resume_till_{i}_rows.csv", index = False, encoding= 'utf-8')
    prev = i

In [ ]:
#created just to check that if there are any resume with null values


# for i in range(100, 1600, 100):
#     temp_df = pd.read_csv(f"resume_till_{i}_rows.csv")
#     print(f"resume till {i}")
#     print(temp_df.isnull().sum());

In [58]:
sample_df.head()

,resume,jd,ats_score
1525,YASH RAJ SUMAN \nYashsuman15@gmail.com | +91-7...,Full job description\nJob Summary:\n\nWe are s...,19.03
1026,CAREER OBJECTIVE\nSecure a responsible career ...,About Warpfy\n\nWe are a fast-growing cross-bo...,36.19
354,JEENAT \n \nharjeenat@gmail.com \n \n \n623960...,About Warpfy\n\nWe are a fast-growing cross-bo...,35.76


In [5]:
dataframe_name = 'DividedDataset/resume_till_100_rows.csv'

In [6]:
temp_df = pd.read_csv(dataframe_name)

In [71]:
temp_df.shape

(101, 3)

In [63]:
atsscore = []

for i in range(102):
    resume = temp_df.iloc[i,0]
    job    = temp_df.iloc[i,1]

    prompt_content = f"""
    You are an expert resume analyst. Your task is to analyze a candidate's resume and a job description to determine their fit. You MUST ONLY provide a JSON object in your response. Do not include any other text or conversational phrases.

    [JOB DESCRIPTION]
    {job}

    [CANDIDATE RESUME]
    {resume}

    Analyze the candidate's skills and experience against the job description.
    Provide a final match score on a scale from 1 to 100, where 1 is no match and 100 is a perfect match.
    

    Format your output strictly as a JSON object with the following keys:
    {{
      "match_score": int,
    }}
    """

    response = ollama.chat(
        model='gemma3',
        messages=[{'role': 'user', 'content': prompt_content}]
    )

    try:
        raw_content = response['message']['content']
        json_str = extract_json(raw_content)   # 🧹 clean response
        ats_score_data = json.loads(json_str)
    
        match_score = ats_score_data.get('match_score', 'N/A')
    
        atscoretemp.append(match_score)
    
        print(i, 'done')
    
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON response at index {i}: {e}")
        print("Raw response:", response['message']['content'])


temp_df['new_ats'] = atscoretemp

0 done
1 done
2 done


In [66]:
temp_df.to_csv(dataframe_name, index= False, encoding= 'utf-8')

In [73]:
new_df = df.iloc[1500: , :]

In [75]:
new_df.to_csv("resume_till_1560_rows", index = False, encoding = 'utf-8')